# Session 5 — Spark: clean, aggregate, join, save

Last session we **found** the problems in the messy listening log. Today we:
1. **Clean** it with Spark — the same rules as session 2, but they would work on a billion rows
2. **Aggregate**: answer questions with `groupBy` + `agg`
3. **Join**: combine plays with tracks, artists and users
4. **Save** results as CSV and Parquet, and see why data engineers prefer Parquet

## Homework review (15 min)
Open `homework/hw04_spark_basics.ipynb`.

---
## Setup

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

In [ ]:
log = spark.read.csv(str(DATA / "messy" / "listening_log_messy.csv"), header=True)
log.show(5, truncate=False)

---
## Part 1 — Cleaning with Spark

Spark has built-in functions for almost every cleaning task. A few you need today:

| Problem | Spark function |
|---|---|
| exact duplicate rows | `df.dropDuplicates()` (or `dropDuplicates(["col"])` for duplicates by key) |
| spaces around text | `F.trim(col)` |
| casing | `F.lower`, `F.upper`, `F.initcap` (= Python's `.title()`) |
| replace text | `F.regexp_replace(col, pattern, replacement)` |
| change type | `col.cast("int")` — gives `null` if the text is not a number |
| if / elif / else | `F.when(cond, value).when(...).otherwise(value)` |
| first non-null value | `F.coalesce(a, b, c)` |
| text → timestamp | `F.to_timestamp(col, "dd/MM/yyyy HH:mm")` — `null` if the format does not match |

Let's clean the names together:

In [ ]:
names_fixed = (
    log
    .dropDuplicates()
    .withColumn("artist", F.initcap(F.trim(F.col("artist"))))
    .withColumn("track", F.initcap(F.trim(F.col("track"))))
)
names_fixed.groupBy("artist").count().orderBy("artist").show(25)

Every artist now appears once with one spelling. Compare: in session 2 this took a function and a loop.

### Exercise 1 (Core) — Country and device
Starting from `names_fixed`, build `step2` where:
- `country` is trimmed and upper-case (`" at "` → `"AT"`)
- `device` is trimmed, lower-case, and spaces are replaced by `_` (`"Smart Speaker"` → `"smart_speaker"`)

In [ ]:
# YOUR CODE HERE

step2.groupBy("country").count().show()
step2.groupBy("device").count().show()  # expected: 4 devices only

### Exercise 2 (Core) — Durations and timestamps
Build `clean_log` from `step2`:

1. `seconds` (int) from `duration`:
   - contains `":"` → minutes * 60 + seconds. `F.split(col, ":")` gives an array; `.getItem(0)` is the first part.
   - ends with `"s"` → remove the `s` (`F.regexp_replace(col, "s$", "")`) and cast to int
   - otherwise → cast to int (`"unknown"` becomes `null` automatically)
2. `played_at` as a real timestamp: try the three formats with `F.coalesce`:
   `"yyyy-MM-dd HH:mm:ss"`, `"dd/MM/yyyy HH:mm"`, `"d MMM yyyy, HH:mm"`
3. `play_id` cast to int; drop the old `duration` column.

In [ ]:
# YOUR CODE HERE

clean_log.printSchema()
clean_log.show(5)
print(clean_log.count(), "rows")  # expected: 600
print(clean_log.filter(F.col("played_at").isNull()).count(), "unparsed timestamps")  # expected: 0

> **Under the hood — why not reuse our Python function?**
> Spark *can* run normal Python functions on every row (a **UDF**, user-defined function):
> ```python
> parse_duration_udf = F.udf(parse_duration, "int")
> log.withColumn("seconds", parse_duration_udf("duration"))
> ```
> It works, but every value must be shipped from Spark's engine (running on the Java Virtual Machine) to a Python process and back.
> Built-in functions run inside the engine and are usually **much** faster. Rule of thumb: built-ins first, UDFs only when there is no other way.

---
## Part 2 — Aggregations

`groupBy(...)` + `agg(...)` = "for each group, calculate ...". Many calculations at once, each with a name (`alias`):

In [ ]:
(
    clean_log
    .groupBy("device")
    .agg(
        F.count("*").alias("plays"),
        F.countDistinct("user").alias("listeners"),
        F.round(F.avg("seconds"), 1).alias("avg_seconds"),
    )
    .orderBy(F.desc("plays"))
    .show()
)

Useful aggregate functions: `F.count`, `F.countDistinct`, `F.sum`, `F.avg`, `F.min`, `F.max`.
They all **ignore nulls** (except `F.count("*")`, which counts rows).

### Exercise 3 (Core) — Top artists again
Compute the **top 5 artists by listening minutes** (`sum(seconds) / 60`, rounded to 1 decimal) and their number of plays.
Do you get the same answer as in session 2, Exercise 7?

In [ ]:
# YOUR CODE HERE

top_artists.show()

**Stretch:** listening minutes per **day** (`F.to_date("played_at")`) and country. Which country listened most on 5 September?

In [ ]:
# YOUR CODE HERE

---
## Part 3 — Joins

The real Wavely event stream only contains **ids** (`user_id`, `track_id`). Names, genres and plans live in separate
**reference** tables. A **join** combines them by matching keys.

```
plays                         tracks                               plays ⨝ tracks
event_id  track_id            track_id  title       genre          event_id  track_id  title       genre
e1        T0028       ⨝       T0028     Neon Rain   pop      =     e1        T0028     Neon Rain   pop
e2        T0128               T0128     Wild Fire   rock           e2        T0128     Wild Fire   rock
```

Let's load the first week of real play events (one file per day — Spark reads them all with a `*` pattern) and the reference tables.

In [ ]:
PLAYS_SCHEMA = (
    "event_id STRING, event_ts TIMESTAMP, user_id STRING, track_id STRING, ms_played INT, "
    "device STRING, country STRING, skipped BOOLEAN, shuffle BOOLEAN"
)
plays = (
    spark.read.csv(str(DATA / "raw" / "plays_2026-09-0[1-7].csv"), header=True, schema=PLAYS_SCHEMA)
    .dropDuplicates(["event_id"])  # the app sometimes sends an event twice
    .filter(F.col("ms_played") >= 0)  # remove broken rows (more on this in session 6)
)

tracks = spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True, inferSchema=True)
artists = spark.read.csv(str(DATA / "reference" / "artists.csv"), header=True, inferSchema=True)
users = spark.read.csv(str(DATA / "reference" / "users.csv"), header=True, inferSchema=True)

print(plays.count(), "plays")
plays.show(3)

In [ ]:
plays_with_tracks = plays.join(tracks, on="track_id", how="inner")
plays_with_tracks.select("event_id", "track_id", "title", "genre", "ms_played").show(5)

Join types you need to know:

| `how=` | keeps |
|---|---|
| `"inner"` | only rows with a match on **both** sides |
| `"left"` | **all** rows of the left table; `null`s where the right side has no match |
| `"left_anti"` | left rows **without** a match — perfect for finding orphans ("plays of tracks we don't know") |

Careful with columns that exist on both sides: `tracks` and `artists` both have a `genre` column, and
`users` and `plays` both have `country`. Select or rename before joining to avoid confusion.

In [ ]:
orphans = plays.join(tracks, on="track_id", how="left_anti")
print(orphans.count(), "plays with an unknown track")

### Exercise 4 (Core) — Skip rate per genre
Which genre gets skipped most? Join `plays` with `tracks` and compute per genre:
`plays` (count) and `skip_rate` = average of `skipped` cast to int, in percent, rounded to 1 decimal. Sort by skip rate.

In [ ]:
# YOUR CODE HERE

skip_by_genre.show()

### Exercise 5 (Core) — Free vs premium
Do premium users listen more? Join `plays` with `users` (only `user_id` and `plan` — use `users.select(...)`) and compute per plan:
- `listeners`: distinct users
- `hours`: total `ms_played` in hours (rounded to 1 decimal)
- `hours_per_listener`: hours / listeners (rounded to 2 decimals)

In [ ]:
# YOUR CODE HERE

by_plan.show()

### Exercise 6 (Stretch) — Top artist per country
Join plays → tracks → artists. For each listener `country`, find the artist with the most plays.
Hint: first count plays per (country, artist_name). Then either use a **window function** (look up `Window.partitionBy(...).orderBy(...)` and `F.row_number()`)
or think of another way.

In [ ]:
# YOUR CODE HERE

---
## Part 4 — Saving results: CSV vs Parquet

Spark writes a **folder** (not one file): one part-file per partition, so every executor can write in parallel.
`mode("overwrite")` replaces old results — so running the cell twice gives the same result.

In [ ]:
OUTPUT = DATA.parent / "output"

clean_log.write.mode("overwrite").option("header", True).csv(str(OUTPUT / "clean_log_csv"))
clean_log.write.mode("overwrite").parquet(str(OUTPUT / "clean_log_parquet"))

for folder in ["clean_log_csv", "clean_log_parquet"]:
    files = [p for p in (OUTPUT / folder).iterdir() if p.name.startswith("part-")]
    size_kb = sum(p.stat().st_size for p in files) / 1024
    print(f"{folder:<20} {len(files)} part file(s), {size_kb:.1f} KB")

Now read both back and compare the schemas:

In [ ]:
spark.read.csv(str(OUTPUT / "clean_log_csv"), header=True).printSchema()
spark.read.parquet(str(OUTPUT / "clean_log_parquet")).printSchema()

| | CSV | Parquet |
|---|---|---|
| readable by humans | yes | no (binary) |
| keeps column types | no — everything is text again | **yes** |
| size | large | small (compressed) |
| read only some columns | must read everything | reads only the columns you need (**columnar**) |
| used for | exchanging data with people / old systems | **data lakes, pipelines, Spark** |

> **Under the hood:** `df.write.partitionBy("country").parquet(path)` creates one sub-folder per country (`country=AT/`, `country=SK/`, ...).
> A query that filters on `country = 'AT'` then only reads that folder. You will use this in session 9.

## Wrap-up

| Concept | Example |
|---|---|
| clean | `dropDuplicates`, `trim`, `initcap`, `regexp_replace`, `cast`, `F.when`, `F.coalesce`, `to_timestamp` |
| aggregate | `groupBy("x").agg(F.sum("y").alias("total"))` |
| join | `a.join(b, on="key", how="left")`, `left_anti` for orphans |
| save | `df.write.mode("overwrite").parquet(path)` |

**Homework:** `homework/hw05_spark_analysis.ipynb`